# Initial-Condition Audit & Campaign Analysis

This notebook covers `workflows/diagnostics/initial_conditions/` — a 6-step IC audit pipeline. It explores differences in initialization states (IC differences) and checks physical consistency between coupled model components.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import os
import sys
import json
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from pathlib import Path

ENVIRONMENT_SHARE = Path(sys.prefix) / 'share'
os.environ.setdefault('PROJ_DATA', str(ENVIRONMENT_SHARE / 'proj'))
os.environ.setdefault('GDAL_DATA', str(ENVIRONMENT_SHARE / 'gdal'))

_repo_override = os.environ.get("ESP_LAB_REPO_ROOT")
_repo_candidates = (
    [Path(_repo_override).expanduser().resolve()]
    if _repo_override
    else [Path.cwd().resolve(), *Path.cwd().resolve().parents]
)
REPO_ROOT = next((p for p in _repo_candidates if (p / "workflows" / "diagnostics" / "initial_conditions").is_dir()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Start Jupyter inside ESP-Lab or set ESP_LAB_REPO_ROOT to its checkout.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

CONFIG_PATH = REPO_ROOT / "workflows" / "diagnostics" / "initial_conditions" / "config.yaml"


## Configuration

In [ ]:
# =========================================================================
# USER CONTROL PANEL
# =========================================================================

# CONFIG_PATH is set above
PILOT_ONLY = True
SINGLE_DATE = None
FILTER_COMPONENTS = None
COMPUTE_HASH = True
FIGURE_OUTDIR = Path("/global/cfs/cdirs/e3sm/www/zhan391/esp-lab_diag/ic_analysis")
FIGURE_OUTDIR.mkdir(parents=True, exist_ok=True)
# =========================================================================

## Step 1 — Inventory & Hash

In [ ]:
%%time
from workflows.diagnostics.initial_conditions.inventory_and_hash import run as run_step1

manifest_df = run_step1(config_path=CONFIG_PATH, pilot_only=PILOT_ONLY, single_date=SINGLE_DATE, compute_hash=COMPUTE_HASH)
display(manifest_df.head())


## Step 2 — Compare NetCDF Structure

In [ ]:
%%time
from workflows.diagnostics.initial_conditions import compare_netcdf_structure

class_df = compare_netcdf_structure.run(config_path=CONFIG_PATH, pilot_only=PILOT_ONLY, single_date=SINGLE_DATE)
display(class_df.head())


## Step 3 — Compute Variable-Level IC Statistics

In [ ]:
%%time
from workflows.diagnostics.initial_conditions import compute_ic_statistics

stats_df = compute_ic_statistics.run(config_path=CONFIG_PATH, pilot_only=PILOT_ONLY, single_date=SINGLE_DATE, filter_components=FILTER_COMPONENTS)
display(stats_df.head())


## Step 4 — Plot Component Differences

In [ ]:
%%time
from workflows.diagnostics.initial_conditions import plot_component_differences

plot_component_differences.run(config_path=CONFIG_PATH, pilot_only=PILOT_ONLY, single_date=SINGLE_DATE, filter_components=FILTER_COMPONENTS, figure_outdir=FIGURE_OUTDIR)
print(f"Output figures saved to {FIGURE_OUTDIR}")


## Step 5 — Cross-Component Physical Consistency

In [ ]:
%%time
from workflows.diagnostics.initial_conditions import check_physical_consistency

consistency_df = check_physical_consistency.run(config_path=CONFIG_PATH, pilot_only=PILOT_ONLY, single_date=SINGLE_DATE)
display(consistency_df.head())


## Step 6 — Campaign Summary

In [ ]:
%%time
from workflows.diagnostics.initial_conditions import summarize_campaign
from IPython.display import Image, display
import yaml

campaign_df = summarize_campaign.run(config_path=CONFIG_PATH, pilot_only=PILOT_ONLY)
display(campaign_df.head())

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)
    
out_root = CONFIG_PATH.parent / cfg.get("output", {}).get("root_dir", "output")
cs_dir = out_root / cfg.get("output", {}).get("subdirs", {}).get("campaign_summary", "campaign_summary")
img_path = cs_dir / "campaign_summary.png"

if img_path.exists():
    display(Image(filename=str(img_path)))


## Validation

In [ ]:
assert not manifest_df.empty, "Step 1 returned empty results"
assert not class_df.empty, "Step 2 returned empty results"
assert not stats_df.empty, "Step 3 returned empty results"
assert not consistency_df.empty, "Step 5 returned empty results"
assert not campaign_df.empty, "Step 6 returned empty results"
print("All validation assertions passed.")
